
# Dataset Merge & Processing (BCODMO annotated)

Purpose: consolidate 2006/2016/2022/2023 seine catch data, standardize taxonomic labels, and produce community/presence tables for downstream CTI analyses. This copy adds step-by-step documentation for BCODMO submission.

**Key inputs (relative to repo root):**
- inp/raw_2006_fish.csv, inputs/raw_2006_invert.csv, inputs/raw_2006_peneid.csv, inputs/2006_site_data.csv
- inputs/raw_2016.csv
- inputs/raw_2022.csv, inputs/raw_2023.csv
- inputs/final_aphia_codex_edited.csv (taxonomic lookup)

**Key outputs:**
- outputs/presence_summary.csv, outputs/presence_pivot.csv
- outputs/presence_summary2.csv, outputs/presence_pivot_merged_sp.csv
- outputs/pivot_all.csv (community matrix used by CTI pipeline)
- outputs/presence_pivot.csv (taxon-by-year presence proportions)

In [1]:

# Environment setup
import sys
import os
import pip

import pandas as pd

print(f"Python executable: {sys.executable}")
print(f"Working directory (before): {os.getcwd()}")

# Set working directory to repository root so relative paths resolve
os.chdir(r"C:/Users/hl51981/OneDrive - University of Georgia/Leavitt_Herbert/PFFW/Manuscripts/Global Change/Revision_repository")
print(f"Working directory (after): {os.getcwd()}")
print(f"pandas version: {pd.__version__}")
print("Python version:", sys.version)
print("\nInstalled packages:")



Python executable: c:\Users\hl51981\.conda\envs\pyo_oracle\python.exe
Working directory (before): c:\Users\hl51981\OneDrive - University of Georgia\Leavitt_Herbert\PFFW\Manuscripts\Global Change\Revision_repository\BCO_DMO uploads\1_raw_data
Working directory (after): C:\Users\hl51981\OneDrive - University of Georgia\Leavitt_Herbert\PFFW\Manuscripts\Global Change\Revision_repository
pandas version: 2.3.1
Python version: 3.13.0 | packaged by Anaconda, Inc. | (main, Oct  7 2024, 21:21:52) [MSC v.1929 64 bit (AMD64)]

Installed packages:


## Load raw inputs

In [ ]:

# Load raw datasets (no transformations yet)
fish_2006 = pd.read_csv("raw_data/raw_2006_fish.csv")
site_data_2006 = pd.read_csv("raw_data/2006_site_data.csv", encoding="latin1")
invert_2006 = pd.read_csv("raw_data/raw_2006_invert.csv")
peneid_2006 = pd.read_csv("raw_data/raw_2006_peneid.csv")
raw_2016 = pd.read_csv("raw_data/raw_2016.csv")
raw_2022 = pd.read_csv("raw_data/raw_2022.csv")
raw_2023 = pd.read_csv("raw_data/raw_2023.csv")

# Taxonomic lookup table (includes code-based names for each survey)
species_list = pd.read_csv("raw_data/final_aphia_codex_edited.csv")
species_list.columns = species_list.columns.str.strip()


## Process 2022 and 2023 data (wide to long, code to valid name)

In [ ]:

# 2022: melt to long format and attach valid taxon names
data_2022 = raw_2022.copy()
data_2022["Year"] = "2022_2023"
data_2022 = data_2022.melt(id_vars=["site_date_key", "Year"], var_name="species_code", value_name="Count")
data_2022 = data_2022.rename(columns={"site_date_key": "SampleID"})
data_2022 = data_2022.merge(species_list[["species_code", "valid_name"]], on="species_code", how="left")
data_2022 = data_2022.rename(columns={"valid_name": "Taxon", "species_code": "species"})
data_2022 = data_2022[["Year", "SampleID", "species", "Taxon", "Count"]]

# 2023: same structure as 2022
data_2023 = raw_2023.copy()
data_2023["Year"] = "2022_2023"
data_2023 = data_2023.melt(id_vars=["site_date_key", "Year"], var_name="species_code", value_name="Count")
data_2023 = data_2023.rename(columns={"site_date_key": "SampleID"})
data_2023 = data_2023.merge(species_list[["species_code", "valid_name"]], on="species_code", how="left")
data_2023 = data_2023.rename(columns={"valid_name": "Taxon", "species_code": "species"})
data_2023 = data_2023[["Year", "SampleID", "species", "Taxon", "Count"]]

# Track union of columns to guarantee alignment before stacking
all_columns = set(data_2022.columns).union(set(data_2023.columns))


## Align column sets and stack 2022?2023 data

In [ ]:

# Add any missing columns (species not seen in one of the years) and fill with zeros, then reorder consistently
for col in all_columns:
    if col not in data_2022.columns:
        data_2022[col] = 0
    if col not in data_2023.columns:
        data_2023[col] = 0

data_2022 = data_2022[sorted(all_columns)]
data_2023 = data_2023[sorted(all_columns)]

combined_202x_data = pd.concat([data_2022, data_2023], ignore_index=True)


## Process 2016 data (Doerr codes to valid names)

In [ ]:

# Map Doerr survey codes to valid names and collapse genus-level Palaemonetes
doerr_to_fullname = species_list.set_index("doerr_name")["valid_name"].to_dict()

data_2016 = raw_2016.copy()
data_2016["Year"] = "2016"
data_2016 = data_2016.drop(columns=["date", "bay"], errors="ignore")
data_2016 = data_2016.melt(id_vars=["site_code", "Year"], var_name="species", value_name="Count")
data_2016 = data_2016.rename(columns={"site_code": "SampleID"})
data_2016["Taxon"] = data_2016["species"].replace(doerr_to_fullname)
data_2016["Taxon"] = data_2016["Taxon"].str.strip()

# Collapse all Palaemonetes observations to genus level for consistent treatment
is_palaemonetes = data_2016["Taxon"].str.startswith("Palaemonetes")
data_2016.loc[is_palaemonetes, "Taxon"] = "Palaemonetes"

data_2016 = (
    data_2016.groupby(["Year", "SampleID", "Taxon", "species"], as_index=False)
    .agg({"Count": "sum"})
)


## Process 2006 data (filter marsh/October, map names)

In [ ]:

# Constrain to marsh habitat in October for 2006 sampling
site_data_2006["Date"] = pd.to_datetime(site_data_2006["Date"], errors="coerce")
marsh_fall_sites = site_data_2006[
    (site_data_2006["GeneralHabitat"].str.strip().str.lower() == "marsh") &
    (site_data_2006["Date"].dt.month == 10)
]
valid_sample_ids = marsh_fall_sites["SampleNumber"].unique()

# Normalize names using 2005 survey codes
fish_2006["Year"] = "2006"
peneid_2006["Year"] = "2006"
invert_2006["Year"] = "2006"
invert_2006["Count"] = 1  # assume presence when not quantified

minello_to_fullname = species_list.set_index("2005_name")["valid_name"].to_dict()

data_2006 = pd.concat([
    fish_2006[["Year", "SampleNumber", "Taxon", "Count"]],
    peneid_2006[["Year", "SampleNumber", "Taxon", "Count"]],
    invert_2006[["Year", "SampleNumber", "Taxon", "Count"]]
], ignore_index=True)

data_2006 = data_2006.rename(columns={"SampleNumber": "SampleID", "Taxon": "species"})
data_2006["Taxon"] = data_2006["species"].replace(minello_to_fullname)

data_2006 = data_2006[data_2006["SampleID"].isin(valid_sample_ids)]


## Combine all years and clean

In [ ]:

all_years = pd.concat([data_2006, data_2016, combined_202x_data], ignore_index=True)
all_years = all_years.dropna(subset=["Taxon"])
all_years = all_years[all_years["Count"] > 0]

species_check = all_years[["Taxon", "species"]].drop_duplicates()


## Summaries: abundance and presence (unfiltered)

In [ ]:

# Total abundance by taxon and year
abundance_summary = (
    all_years.groupby(["Year", "Taxon"], as_index=False)
    .agg(Total_Count=("Count", "sum"))
    .pivot(index="Taxon", columns="Year", values="Total_Count")
    .fillna(0).astype(int).reset_index()
)

# Sampling effort per year
sites_sampled_per_year = pd.DataFrame({
    "Year": ["2006", "2016", "2022_2023"],
    "Unique_Sites_Sampled": [
        data_2006["SampleID"].nunique(),
        data_2016["SampleID"].nunique(),
        combined_202x_data["SampleID"].nunique(),
    ]
})

# Presence counts per taxon-year (number of sites with detections)
presence_summary = (
    all_years.groupby(["Year", "Taxon"])["SampleID"].nunique().reset_index(name="Sites_Present")
)

presence_summary = presence_summary.merge(sites_sampled_per_year, on="Year", how="left")
presence_summary["Proportion_of_Sites"] = (
    presence_summary["Sites_Present"] / presence_summary["Unique_Sites_Sampled"]
).round(3)

presence_summary.to_csv("outputs/presence_summary.csv", index=False)

presence_pivot = (
    presence_summary.pivot(index="Taxon", columns="Year", values="Proportion_of_Sites")
    .fillna(0)
    .reset_index()
)

presence_pivot.to_csv("outputs/presence_pivot.csv", index=False)


## Summaries and Abundance/Presence with Harmonized Species Catagories


In [ ]:

# === Abundance summary table ===
abundance_summary2 = (
	all_years.groupby(["Year", "Taxon"], as_index=False)
	.agg(Total_Count=("Count", "sum"))
	.pivot(index="Taxon", columns="Year", values="Total_Count")
	.fillna(0).astype(int).reset_index()
)

# === Sampling effort by year ===
sites_sampled_per_year2 = pd.DataFrame({
	"Year": ["2006", "2016", "2022_2023"],
	"Unique_Sites_Sampled": [
		data_2006["SampleID"].nunique(),
		data_2016["SampleID"].nunique(),
		combined_202x_data["SampleID"].nunique(),
	]
})

# Collapse select genera/species into analysis-ready bins for CTI

replace_dict = {
    "Minuca": "Minuca spp.",
    "Minuca longisignalis": "Minuca spp.",
    "Minuca pugnax": "Minuca spp.",
    "Minuca rapax": "Minuca spp.",
    "Leander tenuicornis": "Palaemon spp.",
    "Palaemon intermedius": "Palaemon spp.",
    "Palaemon pugio": "Palaemon spp.",
    "Palaemon vulgaris": "Palaemon spp.",
    "Palaemonetes": "Palaemon spp.",
    "Callinectes": "Callinectes sapidus",
    "Alpheus": "Alpheus heterochaelis",
    "Panopeus herbstii": "Panopeus spp.",
    "Panopeus obesus": "Panopeus spp.",
    "Panopeus simpsoni": "Panopeus spp.",
    "Farfantepenaeus": "Penaeus aztecus",
}

# Minuca was not identified to species in 3/4 seasons, but longisignalis is by far the most prevalent at our sites, so we are simplifying all Minuca observations to be longisignalis 
# Palaemon was not identified to species in one of the seasons. Pugio and vulgaris are both common and have similar ranges, so they were compiled for this analysis
# Callinecres, Alpheus, and Farfantepenaus were often identified to genus level in 2005 and 2006. Subsiquent years found that these genuses were overwhelmingly attibuted to one species (> 90%)
# so these genus-level observations were attributed to the most common species in each genus (C. sapidus, A. heterochaelis, Penaus (Farfantepenaus) aztecus)
# Panopeus herbstii was a species complex that was split into Panopeus obesus and Panopeus simpsoni, the new convention was used in 2022 and 2023 but not other years, so P. simpsoni and P. obesus were compiled 
# into Panopeus spp. and analyzed together. These species have similar ranges, but the modern P. herbstii is mostly confined to the atlantic coast. 
# Species will be compiled to their "analysis ready" form here so that statistics can be run on processing outputs. 

all_years["Taxon"] = all_years["Taxon"].replace(replace_dict)

# Collapse duplicates after replacement
collapsed = (
    all_years
    .groupby(["SampleID", "Taxon"], as_index=False)
    .agg({"Count": "sum"})
)

# Community matrix used for CTI calculations
pivot_all = collapsed.pivot(index="SampleID", columns="Taxon", values="Count").fillna(0)
sample_year = all_years[["SampleID", "Year"]].drop_duplicates()
pivot_all = pivot_all.merge(sample_year, on="SampleID", how="left")

pivot_all.to_csv("outputs/pivot_all.csv", index=False)

names_check = pivot_all.columns.tolist()


## Generate Harmonized table for GBIF analysis later

In [ ]:


# === Presence summary: number of sites each taxon was detected in ===
presence_unfiltered= (
	all_years.groupby(["Year", "Taxon"])["SampleID"]
	.nunique()
	.reset_index(name="Sites_Present")
)

merged_df = presence_unfiltered.merge(
    species_list[['valid_name', 'taxonomic_rank']],
    left_on='Taxon',
    right_on='valid_name',
    how='left'
)

# Extract all target taxa names from replace_dict
replace_targets = list(set(replace_dict.values()))

# Modified filter step
filtered_df = merged_df[
    (merged_df['taxonomic_rank'] == 'Species') |
    (merged_df['Taxon'].isin(replace_targets))
]


# === Merge with total number of sites per year ===
presence_summary2 = filtered_df.merge(
	sites_sampled_per_year2, on="Year", how="left"
)

# === Calculate proportion of sites visited with presence ===
presence_summary2["Proportion_of_Sites"] = (
	presence_summary2["Sites_Present"] / presence_summary2["Unique_Sites_Sampled"]
).round(3)
presence_summary2.to_csv("outputs/presence_summary2.csv")

# Check for duplicates in the pivot index and columns
duplicates = presence_summary2.duplicated(subset=["Taxon", "Year"], keep=False)
presence_summary2[duplicates].sort_values(by=["Taxon", "Year"])
# === Pivot for viewing proportions by taxon and year ===
presence_pivot2 = (
	presence_summary2.pivot(index="Taxon", columns="Year", values="Proportion_of_Sites")
	.fillna(0)
	.reset_index()
)

presence_pivot2.to_csv("outputs/presence_pivot_merged_sp.csv")


,Year,Taxon,Sites_Present,valid_name,taxonomic_rank,Unique_Sites_Sampled,Proportion_of_Sites


In [ ]:

# === Pivot for viewing proportions by taxon and year ===
presence_pivot2 = (
	presence_summary2.pivot(index="Taxon", columns="Year", values="Proportion_of_Sites")
	.fillna(0)
	.reset_index()
)

presence_pivot2.to_csv("outputs/presence_pivot_merged_sp.csv")

